# Rapport - Méthode des volumes finis

# 1. Objectif
L’objectif de ce projet est de simuler le transport d’un constituant à travers un labyrinthe en utilisant un champ d’écoulement. Pour cela, on étudie l’évolution de la concentration au cours du temps et sa propagation dans le domaine.

![Labyrinthe étudié](laby.jpg)

Ce phénomène est décrit par l’équation d’advection bidimensionnelle, qui traduit le transport de la concentration sous l’effet du champ de vitesse ((u,v)) :

$
\frac{\partial c}{\partial t}
+\frac{\partial (uc)}{\partial x}
+\frac{\partial (vc)}{\partial y}
=0
$

## 1.1 Simplification de l'équation d'advection
L’équation d’advection peut être exprimée sous une forme conservative, dans laquelle les termes de flux sont écrits sous la forme de produits entre la concentration et les composantes du champ de vitesse. Pour simplifier sa résolution, il est possible de développer ces termes à l’aide de la règle de dérivation d’un produit.

Dans notre problème, l’écoulement est supposé incompressible. La divergence du champ de vitesse est donc nulle :

$
\frac{\partial u}{\partial x}
+
\frac{\partial v}{\partial y}
=0
$

En développant les termes de flux de l’équation d’advection, on obtient :

$
\frac{\partial c}{\partial t}
+\frac{\partial (uc)}{\partial x}
+\frac{\partial (vc)}{\partial y}
=0
$

$
\frac{\partial c}{\partial t}
+u\frac{\partial c}{\partial x}
+c\frac{\partial u}{\partial x}
+v\frac{\partial c}{\partial y}
+c\frac{\partial v}{\partial y}
=0
$

$
\frac{\partial c}{\partial t}
+u\frac{\partial c}{\partial x}
+v\frac{\partial c}{\partial y}
+c\underbrace{\left(
\frac{\partial u}{\partial x}
+\frac{\partial v}{\partial y}
\right)}_{=0}
=0
$

Puisque la divergence du champ de vitesse est nulle, le terme contenant (c) s’annule. L’équation se simplifie alors sous la forme non conservative :

$
\frac{\partial c}{\partial t}
+u\frac{\partial c}{\partial x}
+v\frac{\partial c}{\partial y}
=0
$

# 2.1. Lecture du labyrinthe et conversion en matrice
La première étape consiste à lire l’image du labyrinthe afin de la transformer en une représentation numérique exploitable pour les calculs ultérieurs. Cette opération est réalisée dans la fonction compute() du fichier laplace_only.py.

In [4]:
def compute(filepath:str, fileout:str = 'out'):
    # Import du module PIL pour lire n'importe quelle image
    from PIL import Image
    # Lecture d'une image
    im = Image.open(filepath)

    # Conversion de l'image en matrice Numpy
    array = np.asarray(im)[:,:,0].copy() # uniquement la composante R d'une image RGB --> à particulariser si l'image est pure N&B par exemple

    # La matrice contient des valeurs comprises entre 0 et 255
    
    # Filtrage du 'gris' --> image N&B
    #  seuil arbitraire à 200
    array[np.where(array<=200)]= 0
    array[np.where(array>200)] = 1

    # Calcul du laplacien du potentiel
    domain, pot = laplace_pot(array,
                            outside_value=0,
                            inside_value=1,
                            pot_in=100.,
                            pot_out=-100.)

    fig, ax = plt.subplots(1,1)
    fig.set_size_inches(10,10)
    ax.imshow(pot)
    fig.savefig(fileout+'.png')

    ax.imshow(domain)
    fig.savefig('domain.png')

    np.savetxt(fileout+'_domain.txt', domain, fmt='%i')
    np.savetxt(fileout+'_pot.txt', pot, fmt='%15.10f')
    np.save(fileout+'_pot.npy', pot)


    return domain, pot

 L’image est d’abord ouverte à l’aide de la bibliothèque `PIL`, puis convertie en un tableau NumPy à deux dimensions. Chaque élément de ce tableau correspond à un pixel de l’image. L’indexation `[:, :, 0]` permet de sélectionner la première composante de couleur de chaque pixel, correspondant au rouge dans une image RGB. Les valeurs obtenues sont comprises entre 0 et 255, où 0 correspond au noir et 255 au blanc.

Afin de distinguer les murs des chemins du labyrinthe, un seuil de 200 est appliqué à cette matrice. Les pixels dont la composante rouge est inférieure ou égale à 200 reçoivent la valeur 0, tandis que ceux dont la composante rouge est strictement supérieure à 200 reçoivent la valeur 1. Cette opération, appelée **binarisation**, permet de réduire les différentes nuances de gris à deux valeurs distinctes. Le seuil de 200 constitue un choix arbitraire permettant de séparer les pixels sombres des pixels clairs.

La matrice binaire ainsi obtenue est ensuite transmise à la fonction `laplace_pot()`, qui sera présentée dans la section suivante. Celle-ci permet notamment de construire le domaine de calcul et de déterminer le potentiel de vitesse. Les représentations graphiques du domaine et du potentiel sont également générées par le programme.

## 2.2. Calcul du potentile de vitesse
Une fois l’image convertie en une matrice binaire, la fonction laplace_pot() est appelée afin de calculer le potentiel de vitesse dans le domaine défini.

La fonction reçoit en arguments le domaine provisoire déterminé par compute(), la valeur considérée dans le domaine de calcul qui est 1 et celle hors domaine qui 0, le potentile imposé à l'entrée qui est 100 et celui imposé à la sortie qui est -100. 

In [ ]:
def laplace_pot(domain, outside_value = 0, inside_value = 1, pot_in = 1., pot_out = -1.):

La fonction réduit ensuite le domaine en supprimant les marges inutiles en déterminant les indices i1, i2, j1, j2 correspondant respectivement à l'indice de la première ligne contenant au moins une cellule hors domaine, celui de la dernière ligne, celui de la première colonne et celui de la dernière colonne. 

In [ ]:
    outside= np.where(domain==outside_value)
    i1 = outside[0].min()
    i2 = outside[0].max()
    j1 = outside[1].min()
    j2 = outside[1].max()

    domain=domain[i1:i2+1,j1:j2+1]

La fonction repère à nouveau les cellules hors domaine et celles qui appartiennent au domaine. En effet, le domaine ayant changé de taille lors du recadrage, les anciennes coordonnées ne correspondent plus nécessairement aux bonnes positions dans la nouvelle matrice.

In [ ]:
    outside= np.where(domain==outside_value)
    inside = np.where(domain==inside_value)

Elle détermine le nombre de cellules appartenant au domaine et donc le nombre d'inconnus, étant le potentil, du système linéaire devant être résolue. Les cellules sont égalements numérotées afind de créer une amtrice de numérotation. Cette numératation permet de faire correspondre les coordonnées d'une cellule dans le labyrinthe à la position de son potentiel dans le vecteur x du système Ax=b à résoudre.

In [ ]:
    # Nombre de points de calcul
    nb = len(inside[0])

    # Matrice de numérotation
    numb_vec  = np.arange(0, nb, dtype=np.int32)
    #   Utilisation d'un stockage CSR par facilité --> mais "lent" pour la recherche des voisins (environ 60x plus lent pour l'image d'exemple)
    # numbering = csr((numb_vec, (inside[0], inside[1])), shape=array.shape)
    #
    # Utilisation d'une matrice pleine --> plus "rapide" pour la recherche des voisins mais demande éventuellement plus de mémoire
    numbering = np.zeros(domain.shape, dtype=np.int32)
    numbering[inside] = numb_vec

Les celluels d'entrées et de sorties sont recherchées sur la première et dernière lignes.

In [ ]:
    # Recherche des zones CL
    #  - amont : ligne supérieure
    #  - aval : ligne inférieure
    inlet  = np.where(domain[0,:]==inside_value)[0]
    outlet = np.where(domain[-1,:]==inside_value)[0]

Le vecteur b est construit en lui imposant les conditions d'entrées et de sorties

In [ ]:
    b  = np.zeros(nb)
    b[0:len(inlet)] = pot_in
    b[-len(outlet):]= pot_out

La matrice A du système Ax=b est construite en identifiant les relations entre les cellules du domaine et leurs voisines directes sous formes de listes. Pour chaque cellule intérieure, le code recherche les éventuels voisins situés en haut, en bas, à gauche et à droite. Lorsqu'un voisin appartient également au domaine, un coefficient égal à \(1\) est ajouté dans la matrice à la position correspondant aux deux cellules. Les cellules situées à l'entrée et à la sortie sont traitées séparément afin d'imposer les conditions aux limites associées aux potentiels fixés. Cette procédure permet de construire les coefficients du système d'équations linéaires qui sera ensuite résolu pour déterminer le potentiel de vitesse dans le labyrinthe.

In [ ]:
# Remplissage de la matrice de poids en CSR = stockage compressé par lignes pour matrices creuses
    # *****************************************
    # Création par "List comprehension"
    Aij_inlet  = [ [1., numbering[0 ,j], numbering[0  ,j  ]] for j in list(inlet)]
    Aij_outlet = [ [1., numbering[-1,j], numbering[-1 ,j  ]] for j in list(outlet)]

    Aij_top    = [ [1., numbering[i ,j], numbering[i-1,j  ]] for i,j in tqdm(zip(inside[0], inside[1]), 'A_top')    if i!=0 and i!=domain.shape[0]-1 if domain[i-1,j  ] == inside_value]
    Aij_bottom = [ [1., numbering[i ,j], numbering[i+1,j  ]] for i,j in tqdm(zip(inside[0], inside[1]), 'A_bottom') if i!=0 and i!=domain.shape[0]-1 if domain[i+1,j  ] == inside_value]
    Aij_left   = [ [1., numbering[i ,j], numbering[i  ,j-1]] for i,j in tqdm(zip(inside[0], inside[1]), 'A_left')   if i!=0 and i!=domain.shape[0]-1 if domain[i  ,j-1] == inside_value]
    Aij_right  = [ [1., numbering[i ,j], numbering[i  ,j+1]] for i,j in tqdm(zip(inside[0], inside[1]), 'A_right')  if i!=0 and i!=domain.shape[0]-1 if domain[i  ,j+1] == inside_value]

    def count_neigh(i,j):

La fonction count_neigh comptent les voisins d'une cellule appartenant au domaine. 

In [ ]:
    def count_neigh(i,j):
        count = 0
        if domain[i-1,j] == inside_value:
            count+=1
        if domain[i+1,j] == inside_value:
            count+=1
        if domain[i,j-1] == inside_value:
            count+=1
        if domain[i,j+1] == inside_value:
            count+=1

        return float(count)

Pour chaque cellule intérieure qui n'est ni sur la première ni sur la dernière ligne, le code construit le coefficient diagonal de la celulle centrale égal à l'opposé du nombre de voisins intérieurs.

In [ ]:
    Aij_center = [ [-count_neigh(i,j), numbering[i,j], numbering[i,j]] for i,j in tqdm(zip(inside[0], inside[1]), 'A_center') if i!=0 and i!=domain.shape[0]-1]

Les 7 listes précédentes sont rassemblées en une. A comprend le premier élément de chaque sous liste c'est à dire les coefficients, ii contient les indices des lignes des coefficients et jj le sidnices des colonnes. Création d'une matrice creuse

In [ ]:
    A_parts = [Aij_inlet, Aij_outlet, Aij_left, Aij_right, Aij_bottom, Aij_top, Aij_center]
    A  = [cur[0] for cur_part in A_parts for cur in cur_part]
    ii = [cur[1] for cur_part in A_parts for cur in cur_part]
    jj = [cur[2] for cur_part in A_parts for cur in cur_part]

    # Création de la CSR sur base des vecteurs de valeurs et des indices de position
    csr_a = csr((A,(ii,jj)))

Après aoir créer les matrices A creuse desc oefficients, x des inconnues et b des valeurs connues, on peut enfin résoudre le système Ax = b.

In [ ]:
    # Résolution du système
    x = spsolve(csr_a,b)

Les potentiles contenus dans le vecteur x sont placés dans une matrice de m^me dimensions que domaine pour faciltiter la compréhension et la représentation.

In [ ]:
    pot = np.zeros(domain.shape, dtype=np.float64)
    pot[inside] = x

A chaque cellule est attribuée une valeur qui décrit son rôle : 0 = hors domaine, 1 = domaine et 2 = sorties ou entrées.

In [ ]:
    # Classification du domaine selon la convention
    #   - 0 = hors domaine
    #   - 1 = domaine intérieur
    #   - 2 = conditions limites
    domain[outside] = 0
    domain[inside]  = 1
    domain[0,inlet]   = 2
    domain[-1,outlet]  = 2

Une nouvelle matrice de domaine et potentiel est crée où des deux lignes et deux colonnes de 0 aux extrémités ont été rajoutés.

In [ ]:
    #ajout d'un bord de 0 sur tout le pourtour
    new_dom = np.zeros((domain.shape[0]+2, domain.shape[1]+2), dtype=np.int32)
    new_dom[1:-1,1:-1] = np.int32(domain)
    new_pot = np.zeros((domain.shape[0]+2, domain.shape[1]+2), dtype=np.float64)
    new_pot[1:-1,1:-1] = pot


Enfin les matrices domaines et potentiel finales sont retournées. 

In [ ]:
    return new_dom, new_pot

<table>
  <tr>
    <td align="center">
      <img src="out.png" width="450">
      <br>
    </td>
    <td align="center">
      <img src="domain.png" width="550">
      <br>
    </td>
  </tr>
</table>

# 2.3. Calcul du champ de vitesse
La fonction calculer_vitesses(), définie dans le fichier vitesses.py calcule les composantes du champ de vitesse u et v.

In [ ]:
import numpy as np

def calculer_vitesses(potentiel, domaine, dx, dy):
    u = np.zeros_like(potentiel)
    v = np.zeros_like(potentiel)

    for i in range(1, potentiel.shape[0]-1):
        for j in range(1, potentiel.shape[1]-1):
            if domaine[i, j] == 1 or domaine[i, j] == 2:
                if domaine[i, j-1] != 0:
                    u[i, j] = -(potentiel[i, j] - potentiel[i, j-1]) / dx

    for i in range(1, potentiel.shape[0]-1):
        for j in range(1,potentiel.shape[1]-1):
            if domaine[i, j] == 1 or domaine[i, j] == 2:
                if domaine[i+1, j] != 0:
                    v[i, j] = (potentiel[i, j] - potentiel[i+1, j]) / dy

    return u, v

La fonction possède comme arguments la matrice domaine, potentiel et les pas de discrétisatiosn spatiales dx et dy. Elle intialise les matrices u et v à 0 de même taille que potentiel. Initialiser à 0 permet de n'agir que sur les cellules donc leur composante de vitesse sera nulle.

Elle parcoure tous les indices des lignes et des colonnes à l'exception de ceux corespondant aux lignes et colonnes de 0 ajoutés tous autours. Elle calcule la composante de vitesse horizontale par différences finies décentrée en amont pour les cellules qui font parti du domaine et dont sa voisine de gauche n'est pas un mur. 

$
u_{i,j}=-\frac{\phi_{i,j}-\phi_{i,j-1}}{\Delta x}
$

La convention est telle que u est positif vers la droite.

Pour v, elle parcoure de le même facon la matrice. Elle calcule la composante de vitesse verticale par différences finis décentrée en aval pour les cellules qui font parti du domaine et dont sa voisine du bas n'est pas un mur. 

$
v_{i,j}=\frac{\phi_{i,j}-\phi_{i+1,j}}{\Delta y}
$

La convention est telle que v est positif vers le bas. 

Cette condition traduit l'imperméabilité des parois du labyrinthe : aucun écoulement ne doit traverser les murs.

Les composantes (u) et (v) sont ainsi stockées dans deux matrices distinctes, qui décrivent les vitesses normales aux faces de la grille. Ces matrices seront ensuite utilisées pour calculer les flux de constituant entre les cellules dans le cadre de la méthode des volumes finis.

De par les choix des calculs de u et v, la cellule et ses propriétés peut etre représentée comme ceci. 

![Positions de u et v](cellule.png)

Les composantes de vitesses son tillustrées par afficher_champs_vitesse().

In [ ]:
import matplotlib.pyplot as plt

def afficher_champs_vitesse(u, v, domaine):
    fig, axes = plt.subplots(1, 2, figsize=(16, 7))
    ny, nx = u.shape

    X, Y = np.meshgrid(np.arange(nx), np.arange(ny))

    pas = 10

    # Facteur d'amplification visuelle des flèches
    amplification = 20

    # --------------------------------------------------
    # Champ de vitesse horizontal u
    # --------------------------------------------------
    Xu = X - 0.5
    Yu = Y

    axes[0].imshow(domaine, origin='upper')

    axes[0].quiver(
        Xu[::pas, ::pas],
        Yu[::pas, ::pas],
        amplification * u[::pas, ::pas],
        np.zeros_like(u[::pas, ::pas]),
        color='red',
        angles='xy',
        scale_units='xy',
        scale=1,
        width=0.003,
        headwidth=5,
        headlength=7
    )

    axes[0].set_xticks(np.arange(-0.5, nx, 1), minor=True)
    axes[0].set_yticks(np.arange(-0.5, ny, 1), minor=True)
    axes[0].grid(
        which='minor',
        linestyle='-',
        linewidth=0.5,
        color='black',
        alpha=0.5
    )

    axes[0].set_title("Champ de la composante de vitesse horizontale u")
    axes[0].set_xlabel("indices colonnes")
    axes[0].set_ylabel("indices lignes")

    # --------------------------------------------------
    # Champ de vitesse vertical v
    # --------------------------------------------------
    Xv = X
    Yv = Y + 0.5

    axes[1].imshow(domaine, origin='upper')

    axes[1].quiver(
        Xv[::pas, ::pas],
        Yv[::pas, ::pas],
        np.zeros_like(v[::pas, ::pas]),
        amplification * v[::pas, ::pas],
        color='blue',
        angles='xy',
        scale_units='xy',
        scale=1,
        width=0.003,
        headwidth=5,
        headlength=7
    )

    axes[1].set_xticks(np.arange(-0.5, nx, 1), minor=True)
    axes[1].set_yticks(np.arange(-0.5, ny, 1), minor=True)
    axes[1].grid(
        which='minor',
        linestyle='-',
        linewidth=0.5,
        color='black',
        alpha=0.5
    )

    axes[1].set_title("Champ de la composante de vitesse verticale v")
    axes[1].set_xlabel("indices colonnes")
    axes[1].set_ylabel("indices lignes")

    plt.tight_layout()
    plt.savefig("champs_vitesse.png", dpi=300, bbox_inches="tight")
    plt.show()

<img src="champs_vitesse.png" alt="Champ de u et v" width="750">

## 2.4. Vérification de la divergence nulle
La focntion vérification_divergence du fichier divergence calcule la divergence du champ de vitese et vérifie qu'elle nulle.

In [3]:
import numpy as np

def verification_divergence(u, v, domaine, dx, dy):

    divergence = np.zeros_like(u)

    for i in range(2, u.shape[0]-2):
        for j in range(1, u.shape[1]-1):
            if domaine[i, j] != 0:

                du_dx = (u[i, j+1] - u[i, j]) / dx
                dv_dy = (v[i, j] - v[i-1, j]) / dy
                divergence[i, j] = du_dx + dv_dy

    print("Divergence maximale :", np.max(np.abs(divergence)))

    return divergence

La fonction prend en arguments la composante horizontale et verticale du champ de vitesse, le domaine, le potentiel ainsi que le spas de discrétisations spatiales. Elle initialise la matrice divergence à 0 afin que toutes els cellules hors du domaine valent déjà 0. Elle parcourue les lignes à partir de la ligne d'indice 2 jusqu'à l'avant avant dernière. En effet, le calcul de dv_dy prenant en compte v[i-1;j], la divergence d'une cellule de ligne d'indice 1 engendrait inévtablement une divergence non-nulle et fausserait alros al vérification. Pas besoind e prendre de tel précaution pour le parcours des colonnes qui va de la colonne d'indice 1 jusqu'a -1.
On obtient une divergence maximale de $3,13 \cdot 10^{13}$ m/s ce qui est suffisament proche de 0 pour conclure que le calcule du champ de vitesse est correcte mais montre que la méthode par différecnes finis n'est pas exacte. 